Step 1 — Install & Setup

In [57]:
!pip install huggingface_hub openai -q

import os
import subprocess
import tempfile
import re
import time
from openai import OpenAI
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

Step 2 — Model Setup & API Connection

In [58]:
HF_API_KEY = "Paste_HF_Token"
MODEL  = "meta-llama/Llama-3.1-8B-Instruct"
PROVIDER = "cerebras"
client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_API_KEY)

Step 2 — Language Config

In [59]:
LANGUAGE_CONFIG = {
    "python":     {"extension": ".py",   "run_cmd": ["python3"], "compile_cmd": None,          "label": "Python"},
    "javascript": {"extension": ".js",   "run_cmd": ["node"],    "compile_cmd": None,          "label": "JavaScript (Node.js)"},
    "java":       {"extension": ".java", "run_cmd": ["java"],    "compile_cmd": ["javac"],      "label": "Java"},
    "c":          {"extension": ".c",    "run_cmd": ["./a.out"], "compile_cmd": ["gcc"],        "label": "C"},
    "cpp":        {"extension": ".cpp",  "run_cmd": ["./a.out"], "compile_cmd": ["g++"],        "label": "C++"},
    "bash":       {"extension": ".sh",   "run_cmd": ["bash"],    "compile_cmd": None,          "label": "Bash"},
    "sql":        {"extension": ".sql",  "run_cmd": ["sqlite3"], "compile_cmd": None,          "label": "SQL"},
    "html":       {"extension": ".html", "run_cmd": None,        "compile_cmd": None,          "label": "HTML/CSS/JS"},
    "r":          {"extension": ".R",    "run_cmd": ["Rscript"], "compile_cmd": None,          "label": "R"},
    "typescript": {"extension": ".ts",   "run_cmd": None,        "compile_cmd": ["npx","tsc"], "label": "TypeScript"},
}

Step 5 — Language Auto-Detection Function

In [60]:
def detect_language(task):
  keywords = {
      "html":       ["html","webpage","website","web page","css","form","ui","frontend", "landing page","button","navbar","table","web app","react"],
      "java":       ["java","spring","maven","android"],
      "javascript": ["javascript","js","node","nodejs","express","json api"],
      "cpp":        ["c++","cpp"],
      "c":          [" c ","c program","c language"],
      "bash":       ["bash","shell script","shell","linux command","terminal script"],
      "sql":        ["sql","database query","select from","sqlite","mysql query"],
      "r":          [" r ","ggplot","tidyverse","r language","data science in r"],
      "typescript": ["typescript",".ts","typed javascript"],
  }
  task_lower = task.lower()
  for lang, keys in keywords.items():
    for k in keys:
      if k in task_lower:
        return lang
  return "python"

Step 6 — Runtime Availability Checker

In [61]:
def check_runtime_available(language):
    checks = {
        "javascript": ["node","--version"],
        "java":       ["java","--version"],
        "c":          ["gcc","--version"],
        "cpp":        ["g++","--version"],
        "r":          ["Rscript","--version"],
    }
    if language not in checks:
        return True
    try:
        result = subprocess.run(checks[language], capture_output=True, timeout=5)
        return result.returncode == 0
    except:
        return False

Step 7 — Code Generation Function (LLM Call)

In [62]:
def generate_code(task, language, previous_code=None, error=None):
    lang_cfg   = LANGUAGE_CONFIG.get(language, LANGUAGE_CONFIG["python"])
    lang_label = lang_cfg["label"]

    html_note = ""
    if language == "html":
        html_note = """For HTML/CSS/JS tasks:
- Write a SINGLE complete .html file with all CSS inside <style> tags and all JS inside <script> tags
- Make it visually polished with proper styling
- Do NOT use external CDN links that may fail — use inline styles and vanilla JS only
- The page must be fully functional as a standalone file"""

    if error and previous_code:
        system_message = f"""You are an expert {lang_label} programmer.
Fix the broken {lang_label} code given the error message.
{html_note}
Return ONLY raw {lang_label} code. No markdown, no code fences, no explanations."""

        user_message = f"""Fix this {lang_label} code:

CODE:
{previous_code}

ERROR:
{error}

Return only the corrected {lang_label} code."""

    else:
        system_message = f"""You are an expert {lang_label} programmer.
Write complete, working {lang_label} code for the given task.
{html_note}
RULES:
- Return ONLY raw {lang_label} code — no markdown, no code fences, no explanations
- Code must be complete and immediately runnable
- Include all necessary imports, functions, and a main entry point
- Output must be clear and well-labelled"""

        user_message = f"""Write {lang_label} code for this task:

{task}

Return only the {lang_label} code."""

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=f"{MODEL}:{PROVIDER}",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": user_message}
                ],
                temperature=0.2,
                max_tokens=4096,
            )
            raw = response.choices[0].message.content.strip()
            raw = re.sub(r"^```[\w]*\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"```\s*$",      "", raw, flags=re.MULTILINE)
            return raw.strip()

        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "rate" in err_str.lower():
                print(f"Rate limited, waiting 20s... (attempt {attempt+1}/3)")
                time.sleep(20)
            elif "503" in err_str or "loading" in err_str.lower():
                print(f"Model loading, waiting 15s... (attempt {attempt+1}/3)")
                time.sleep(15)
            else:
                print(f"API error: {err_str[:200]}")
                if attempt == 2:
                    raise
                time.sleep(5)

    raise Exception("HF API failed after 3 retries")

Step 8 — Code Execution Function

In [63]:
def execute_code(code, language):
    if language == "html":
        return True, code, None, True

    lang_cfg  = LANGUAGE_CONFIG.get(language, LANGUAGE_CONFIG["python"])
    ext       = lang_cfg["extension"]

    with tempfile.NamedTemporaryFile(mode='w', suffix=ext,
                                     delete=False, dir='/tmp') as f:
        f.write(code)
        temp_path = f.name

    try:
        if language == "java":
            match      = re.search(r'public\s+class\s+(\w+)', code)
            class_name = match.group(1) if match else "Main"
            java_path  = f"/tmp/{class_name}.java"
            with open(java_path, 'w') as f:
                f.write(code)
            cr = subprocess.run(["javac", java_path],
                                capture_output=True, text=True, timeout=30)
            if cr.returncode != 0:
                return False, "", f"Compilation error:\n{cr.stderr}", False
            rr = subprocess.run(["java", "-cp", "/tmp", class_name],
                                capture_output=True, text=True, timeout=30)
            if rr.returncode == 0:
                return True, rr.stdout.strip(), None, False
            return False, "", rr.stderr.strip(), False

        elif language in ["c", "cpp"]:
            compiler = "gcc" if language == "c" else "g++"
            out_path = "/tmp/a.out"
            cr = subprocess.run([compiler, temp_path, "-o", out_path],
                                capture_output=True, text=True, timeout=30)
            if cr.returncode != 0:
                return False, "", f"Compilation error:\n{cr.stderr}", False
            rr = subprocess.run([out_path],
                                capture_output=True, text=True, timeout=30)
            if rr.returncode == 0:
                return True, rr.stdout.strip(), None, False
            return False, "", rr.stderr.strip(), False

        elif language == "sql":
            proc = subprocess.run(["sqlite3"], input=code,
                                  capture_output=True, text=True, timeout=30)
            if proc.stderr and not proc.stdout:
                return False, "", proc.stderr.strip(), False
            return True, proc.stdout.strip() or "(Query executed, no output)", None, False

        else:
            run_cmd = lang_cfg["run_cmd"] + [temp_path]
            result  = subprocess.run(run_cmd, capture_output=True,
                                     text=True, timeout=30)
            stdout  = result.stdout.strip()
            stderr  = result.stderr.strip()
            if result.returncode == 0:
                return True, stdout, None, False
            return False, stdout, stderr, False

    except subprocess.TimeoutExpired:
        return False, "", "Error: Execution timed out (30s limit)", False
    except FileNotFoundError as e:
        return False, "", f"Runtime not found: {str(e)}", False
    except Exception as e:
        return False, "", f"Unexpected error: {str(e)}", False
    finally:
        try:
            os.unlink(temp_path)
        except:
            pass

Step 9 — HTML Preview Renderer

In [64]:
def render_html_output(html_code):
    display(HTML(f"""
    <div style="border:2px solid #4CAF50; border-radius:8px;
                overflow:hidden; margin:10px 0;">
        <div style="background:#4CAF50; color:white; padding:8px 14px;
                    font-family:monospace; font-size:13px;">
            LIVE UI PREVIEW
        </div>
        <iframe srcdoc="{html_code.replace('"','&quot;').replace("'", '&#39;')}"
                style="width:100%; height:500px; border:none; background:white;"
                sandbox="allow-scripts allow-forms">
        </iframe>
    </div>
    """))

Step 10 — Save Code to File

In [65]:
def save_code(code, language):
    ext      = LANGUAGE_CONFIG.get(language, {}).get("extension", ".txt")
    filename = f"agent_output{ext}"
    with open(filename, 'w') as f:
        f.write(code)
    print(f"Saved to: {filename}")
    return filename

Step 11 — Main Agent Runner Function

In [66]:
def run_agent(task, language="auto", max_attempts=5):
    if language == "auto":
        language = detect_language(task)

    lang_label = LANGUAGE_CONFIG.get(language, {}).get("label", language)

    print(f"Task     : {task}")
    print(f"Language : {lang_label}")

    if language not in ["python", "html", "sql"] and \
       not check_runtime_available(language):
        print(f"{lang_label} not found — falling back to Python")
        language   = "python"
        lang_label = "Python"

    current_code = None
    last_error   = None

    for attempt in range(1, max_attempts + 1):
        print(f"\nAttempt {attempt}/{max_attempts}")

        if attempt == 1:
            print("Generating code...")
            current_code = generate_code(task, language)
        else:
            print("Fixing error...")
            current_code = generate_code(task, language,
                                          previous_code=current_code,
                                          error=last_error)

        print(f"\n{lang_label.upper()} CODE:")
        print(current_code)

        success, output, error, is_html = execute_code(current_code, language)

        if success:
            if is_html:
                render_html_output(output)
            else:
                if output:
                    print("\nOUTPUT:")
                    print(output)
            save_code(current_code, language)
            print(f"\nDone in {attempt} attempt(s)!")
            return current_code, output
        else:
            last_error = error
            print(f"\nError: {error}")
            if attempt < max_attempts:
                print("Retrying with fix...")

    print("\nAgent failed after all attempts")
    return None, None

Step 12 — Sample Run: Python (Statistics)

In [67]:
run_agent("Create a list of 10 random numbers, find mean median mode, "
          "show largest and smallest with clear labels");

Task     : Create a list of 10 random numbers, find mean median mode, show largest and smallest with clear labels
Language : Python

Attempt 1/5
Generating code...

PYTHON CODE:
import random
import statistics

def generate_random_numbers(n):
    return [random.randint(1, 100) for _ in range(n)]

def calculate_statistics(numbers):
    mean = statistics.mean(numbers)
    median = statistics.median(numbers)
    mode = statistics.mode(numbers)
    return mean, median, mode

def find_largest_smallest(numbers):
    largest = max(numbers)
    smallest = min(numbers)
    return largest, smallest

def main():
    numbers = generate_random_numbers(10)
    print("Random Numbers:")
    print(numbers)
    
    mean, median, mode = calculate_statistics(numbers)
    print("\nStatistics:")
    print(f"Mean: {mean}")
    print(f"Median: {median}")
    print(f"Mode: {mode}")
    
    largest, smallest = find_largest_smallest(numbers)
    print("\nLargest and Smallest:")
    print(f"Largest: {largest}")

Step 13 — Sample Run: HTML Report Card Dashboard

In [68]:
run_agent("Build an HTML dashboard showing a student report card with "
          "5 subjects, their marks, grade, pass/fail status, "
          "and a colorful summary table");

Task     : Build an HTML dashboard showing a student report card with 5 subjects, their marks, grade, pass/fail status, and a colorful summary table
Language : HTML/CSS/JS

Attempt 1/5
Generating code...

HTML/CSS/JS CODE:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Student Report Card</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f0f0f0;
        }
        
        .container {
            width: 80%;
            margin: 40px auto;
            background-color: #fff;
            padding: 20px;
            border: 1px solid #ddd;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
        }
        
        .subject {
            display: flex;
            align-items: center;
            margin-bottom: 20px;
        }
        
        .subject .name {
            width: 30%;
       

Subject,Marks,Grade,Status
Mathematics,85,B,Pass
Science,90,A,Pass
English,78,C,Fail
History,92,A,Pass
Geography,88,B,Pass


Saved to: agent_output.html

Done in 1 attempt(s)!


Step 14 — Sample Run: JavaScript Linked List

In [69]:
run_agent("Write a JavaScript program that implements a linked list "
          "with insert, delete, and display operations, then demonstrate it");

Task     : Write a JavaScript program that implements a linked list with insert, delete, and display operations, then demonstrate it
Language : Java

Attempt 1/5
Generating code...

JAVA CODE:
// Node class representing each element in the linked list
class Node {
    int data;
    Node next;

    public Node(int data) {
        this.data = data;
        this.next = null;
    }
}

// LinkedList class with insert, delete, and display operations
class LinkedList {
    Node head;

    public LinkedList() {
        this.head = null;
    }

    // Insert a new node at the end of the linked list
    public void insert(int data) {
        Node newNode = new Node(data);
        if (head == null) {
            head = newNode;
        } else {
            Node lastNode = head;
            while (lastNode.next != null) {
                lastNode = lastNode.next;
            }
            lastNode.next = newNode;
        }
    }

    // Delete a node with the given data from the linked list
    pu

Step 15 — Sample Run: SQL Students Table

In [70]:
run_agent("Write SQL to create a students table, insert 5 records, "
          "then query to find average marks and top student");

Task     : Write SQL to create a students table, insert 5 records, then query to find average marks and top student
Language : HTML/CSS/JS

Attempt 1/5
Generating code...

HTML/CSS/JS CODE:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Student Table</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f0f0f0;
        }
        
        table {
            border-collapse: collapse;
            width: 50%;
            margin: 20px auto;
        }
        
        th, td {
            border: 1px solid #ddd;
            padding: 10px;
            text-align: left;
        }
        
        th {
            background-color: #f0f0f0;
        }
        
        .container {
            max-width: 800px;
            margin: 40px auto;
            padding: 20px;
            background-color: #fff;
            border: 1px solid #d

Student ID,Name,Marks


Saved to: agent_output.html

Done in 1 attempt(s)!


Step 16 — Interactive UI Widget

In [71]:
LANGUAGES = [
    ("Auto-detect",                    "auto"),
    ("Python",                          "python"),
    ("HTML/CSS/JS (with live preview)", "html"),
    ("Java",                             "java"),
    ("JavaScript (Node.js)",             "javascript"),
    ("C++",                              "cpp"),
    ("C",                                "c"),
    ("SQL",                              "sql"),
    ("Bash",                             "bash"),
    ("R",                                "r"),
]
title_html = widgets.HTML("""
<div style="background:linear-gradient(135deg,#1a1a2e,#16213e);
            padding:20px 24px; border-radius:12px; margin-bottom:12px;">
  <h2 style="color:#e2e8f0; margin:0; font-family:monospace;">
    Autonomous Coding Agent
  </h2>
  <p style="color:#94a3b8; margin:6px 0 0; font-size:13px;">
    · Multi-language · UI preview
  </p>
</div>
""")

task_box = widgets.Textarea(
    placeholder='Describe your coding task in plain English...',
    layout=widgets.Layout(width='100%', height='130px')
)

attempts_slider = widgets.IntSlider(
    value=5, min=1, max=8,
    description='Max fix attempts:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

run_btn   = widgets.Button(description='Run Agent', button_style='success', layout=widgets.Layout(width='160px', height='42px'))
clear_btn = widgets.Button(description='Clear',     button_style='warning', layout=widgets.Layout(width='100px', height='42px'))
status_bar   = widgets.HTML(value='<p style="color:#64748b;font-size:13px;margin:8px 0;">Ready.</p>')

output_panel = widgets.Output(
    layout=widgets.Layout(border='1px solid #334155', border_radius='8px',
                          padding='12px', min_height='100px',
                          max_height='800px', overflow_y='auto')
)

def on_run(b):
  task = task_box.value.strip()
  if not task:
    status_bar.value = '<p style="color:#f87171;">Please enter a task first.</p>'
    return
  run_btn.disabled    = True
  run_btn.description = 'Running...'
  status_bar.value    = '<p style="color:#34d399;">Agent running...</p>'
  with output_panel:
    clear_output(wait=True)
    run_agent(task, language=lang_dropdown.value, max_attempts=attempts_slider.value)
  run_btn.disabled    = False
  run_btn.description = 'Run Agent'
  status_bar.value    = '<p style="color:#64748b;">Done.</p>'

def on_clear(b):
  with output_panel:
    clear_output()
  task_box.value   = ''
  status_bar.value = '<p style="color:#64748b;">Cleared.</p>'
run_btn.on_click(on_run)
clear_btn.on_click(on_clear)

ui = widgets.VBox([
    title_html, task_box, lang_dropdown, attempts_slider,
    widgets.HBox([run_btn, clear_btn], layout=widgets.Layout(gap='10px', margin='8px 0')),
    status_bar, output_panel
], layout=widgets.Layout(padding='8px'))

display(ui)